In [ ]:
#| hide
#| default_exp helpers

In [ ]:
#| export
import functools
class TrackedList(list):
    def __init__(self, iterable):
        super().__init__(iterable)
        self.swaps = 0

    def __setitem__(self, index, value):
        self.swaps += 1
        super().__setitem__(index, value)

class TrackedPredicate:
    def __init__(self, pred):
        self.pred = pred
        self.compares = 0

    def __call__(self, val):
        self.compares += 1
        return self.pred(val)

def profile_partition(func):
    @functools.wraps(func)
    def wrapper(arr, pred, *args, **kwargs):
        tracked_arr = TrackedList(arr)
        tracked_pred = TrackedPredicate(pred)

        result = func(tracked_arr, tracked_pred, *args, **kwargs)

        arr[:] = tracked_arr
        actual_swaps = tracked_arr.swaps // 2

        print(f"[{func.__name__}] Comparisons: {tracked_pred.compares} | Swaps: {actual_swaps}")
        return result
    return wrapper

def profile(func):
    @functools.wraps(func)
    def wrapper(arr, *args, **kwargs):
        tracked = TrackedList(arr)
        preds = []
        wrapped = []
        for arg in args:
            if callable(arg):
                pred = TrackedPredicate(arg)
                preds.append(pred)
                wrapped.append(pred)
            else:
                wrapped.append(arg)
        result = func(tracked, *wrapped, **kwargs)
        arr[:] = tracked
        compares = sum(p.compares for p in preds)
        print(f"[{func.__name__}] Comparisons: {compares} | Swaps: {tracked.swaps // 2}")
        return result
    return wrapper